# Connecting Python to SQL Lab

## Introduction

Welcome to the Connecting Python to SQL lab!

In this lab, you will be working with the Sakila database on movie rentals. Specifically, you will be practicing how to do basic SQL queries using Python. By connecting Python to SQL, you can leverage the power of both languages to efficiently manipulate and analyze large datasets.

Throughout this lab, you will practice how to use Python to retrieve and manipulate data stored in the Sakila database using SQL queries. Let's get started!

## Challenge

In this lab, the objective is to identify the customers who were active in both May and June, and how did their activity differ between months.

To achieve this, follow these steps:

## Step 1: Establish a connection between Python and the Sakila database

Establish a connection between Python and the Sakila database.

In [7]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy import text

load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

In [3]:
connection_string = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine = create_engine(connection_string)

## Step 2: Create the `rentals_month` function

Write a Python function called `rentals_month` that retrieves rental data for a given month and year, passed as parameters, from the Sakila database as a Pandas DataFrame.

The function should take in three parameters:

- `engine`: an object representing the database connection engine to be used to establish a connection to the Sakila database.
- `month`: an integer representing the month for which rental data is to be retrieved.
- `year`: an integer representing the year for which rental data is to be retrieved.

The function should execute a SQL query to retrieve the rental data for the specified month and year from the `rental` table in the Sakila database, and return it as a pandas DataFrame.

In [5]:
def rentals_month(engine, month, year):
    start_date = f"{year}-{month:02d}-01"
    end_date = pd.to_datetime(start_date) + pd.DateOffset(months=1)
    end_date = end_date.strftime("%Y-%m-%d")

    query = text("""
        SELECT *
        FROM rental
        WHERE rental_date >= :start_date
          AND rental_date < :end_date;
    """)

    rentals_df = pd.read_sql(
        query,
        engine,
        params={
            "start_date": start_date,
            "end_date": end_date
        }
    )

    return rentals_df

In [8]:
may_rentals = rentals_month(engine, 5, 2005)

may_rentals.head()

,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1,2005-05-24 22:53:30,367,130,2005-05-26 22:04:30,1,2006-02-15 21:30:53
1,2,2005-05-24 22:54:33,1525,459,2005-05-28 19:40:33,1,2006-02-15 21:30:53
2,3,2005-05-24 23:03:39,1711,408,2005-06-01 22:12:39,1,2006-02-15 21:30:53
3,4,2005-05-24 23:04:41,2452,333,2005-06-03 01:43:41,2,2006-02-15 21:30:53
4,5,2005-05-24 23:05:21,2079,222,2005-06-02 04:33:21,1,2006-02-15 21:30:53


## Step 3: Create the `rental_count_month` function

Develop a Python function called `rental_count_month` that takes the DataFrame provided by `rentals_month` as input along with the month and year and returns a new DataFrame containing the number of rentals made by each `customer_id` during the selected month and year.

The function should also include the month and year as parameters and use them to name the new column according to the month and year.

For example, if the input month is `05` and the year is `2005`, the column name should be `"rentals_05_2005"`.

Hint: Consider making use of pandas `groupby()`.

In [9]:
def rental_count_month(rentals_df, month, year):
    column_name = f"rentals_{month:02d}_{year}"

    rentals_count_df = (
        rentals_df
        .groupby("customer_id")
        .size()
        .reset_index(name=column_name)
    )

    return rentals_count_df

In [10]:
may_count = rental_count_month(may_rentals, 5, 2005)

may_count.head()

,customer_id,rentals_05_2005
0,1,2
1,2,1
2,3,2
3,5,3
4,6,3


In [13]:
may_count.shape

(520, 2)

In [14]:
june_rentals = rentals_month(engine, 6, 2005)

june_count = rental_count_month(june_rentals, 6, 2005)

june_count.head()

,customer_id,rentals_06_2005
0,1,7
1,2,1
2,3,4
3,4,6
4,5,5


In [15]:
june_count.shape

(590, 2)

## Step 4: Create the `compare_rentals` function

Create a Python function called `compare_rentals` that takes two DataFrames as input containing the number of rentals made by each customer in different months and years.

The function should return a combined DataFrame with a new `difference` column, which is the difference between the number of rentals in the two months.

In [16]:
def compare_rentals(df1, df2):
    comparison_df = pd.merge(
        df1,
        df2,
        on="customer_id",
        how="inner"
    )

    first_month_col = df1.columns[1]
    second_month_col = df2.columns[1]

    comparison_df["difference"] = (
        comparison_df[second_month_col] - comparison_df[first_month_col]
    )

    return comparison_df

In [17]:
comparison_df = compare_rentals(may_count, june_count)

comparison_df.head()

,customer_id,rentals_05_2005,rentals_06_2005,difference
0,1,2,7,5
1,2,1,1,0
2,3,2,4,2
3,5,3,5,2
4,6,3,4,1


In [18]:
display(comparison_df.sort_values("difference", ascending=False).head())
display(comparison_df.sort_values("difference", ascending=True).head())

,customer_id,rentals_05_2005,rentals_06_2005,difference
386,454,1,10,9
389,457,1,9,8
178,213,1,9,8
248,295,1,9,8
23,27,1,8,7


,customer_id,rentals_05_2005,rentals_06_2005,difference
173,207,6,1,-5
230,274,6,2,-4
210,250,5,1,-4
167,198,5,1,-4
509,596,6,2,-4


In [19]:
comparison_df.shape

(512, 4)

## Conclusions

After comparing May and June 2005, we can see that rental activity was higher in June.

In May, there were **520 active customers**, while in June there were **590 active customers**. When comparing both months together, **512 customers were active in both May and June**. This means that most of the customers who rented movies in May also rented again in June.

To compare the activity between both months, I created the "difference" column as:

"rentals_06_2005" - "rentals_05_2005"

This helped me understand whether each customer rented more, less, or the same number of movies in June compared to May.

A positive value means that the customer rented more movies in June, a value of zero means that the activity stayed the same, and a negative value means that the customer rented fewer movies in June.

Looking at the results, some customers clearly increased their activity in June. For example, a few customers went from only one rental in May to nine or ten rentals in June. There were also some customers who rented less in June, but overall the results suggest that June was a stronger month for rentals.

In conclusion, the main finding is that customer activity increased from May to June, both in terms of the number of active customers and the rental activity of many customers who were active in both months.
